# Perturbation overlap with measured genes

This notebook checks how many perturbation labels in `adata.obs['perturbation']` are also present in `adata.var_names` for `merged.h5ad`.

Run this notebook with the Python environment at `/projects/b1042/GoyalLab/jaekj/python/KS_perturb/bin/python`.

Computed result from that environment:

- Cells: 197249
- Genes in `adata.var`: 13992
- Unique perturbations in `adata.obs['perturbation']`: 98
- Perturbations also present in `adata.var_names`: 51
- Perturbations missing from `adata.var_names`: 47
- Percent overlap: 52.04%
- `adata.obs['perturbation_type']` is entirely `CRISPRi` in this file.

In [1]:
import sys
from pathlib import Path

import anndata as ad
import pandas as pd

expected_python = Path('/projects/b1042/GoyalLab/jaekj/python/KS_perturb/bin/python')
h5ad_path = Path('/projects/b1042/GoyalLab/jaekj/KeepingScore/perturb-seq/ExPert/anndata/merged.h5ad')
2
print(sys.executable)
assert Path(sys.executable).resolve() == expected_python.resolve(), (
    f'Please run this notebook with {expected_python}; current kernel is {sys.executable}'
)

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/projects/b1042/GoyalLab/jaekj/python/KS_perturb/lib/python3.10/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/projects/b1042/GoyalLab/jaekj/python/KS_perturb/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/projects/b1042/GoyalLab/jaekj/python/KS_perturb/lib/python3.10/site-packages/ipykernel/kernelbase.py", line 340, in dispatch_control
    async with self._control_lock:
  File "/projects/b1042/GoyalLab/jaekj/python/KS_perturb/lib/python3.10/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
  File "/projects/b1042/GoyalLab/jaekj/python/KS_perturb/lib/p

/projects/b1042/GoyalLab/jaekj/python/KS_perturb/bin/python


In [2]:
adata = ad.read_h5ad(h5ad_path, backed='r')
adata

AnnData object with n_obs × n_vars = 197249 × 13992 backed at '/projects/b1042/GoyalLab/jaekj/KeepingScore/perturb-seq/ExPert/anndata/merged.h5ad'
    obs: 'perturbation', 'efficiency', 'dataset', 'eff_score', 'perturbation_type', 'cls_label', 'split'
    obsm: 'ExPert_latent_z_shared'

In [3]:
perturbations = pd.Index(adata.obs['perturbation'].dropna().astype(str).unique(), name='perturbation')
genes = pd.Index(adata.var_names.astype(str), name='gene')

overlap = perturbations.intersection(genes).sort_values()
missing_from_var = perturbations.difference(genes).sort_values()

summary = pd.DataFrame(
    {
        'metric': [
            'cells',
            'genes_in_var',
            'unique_obs_perturbations',
            'perturbations_found_in_var',
            'perturbations_missing_from_var',
            'percent_perturbations_found_in_var',
        ],
        'value': [
            adata.n_obs,
            adata.n_vars,
            len(perturbations),
            len(overlap),
            len(missing_from_var),
            round(100 * len(overlap) / len(perturbations), 2),
        ],
    }
)
summary

,metric,value
0,cells,197249.00
1,genes_in_var,13992.00
2,unique_obs_perturbations,98.00
3,perturbations_found_in_var,51.00
4,perturbations_missing_from_var,47.00
5,percent_perturbations_found_in_var,52.04


In [4]:
pd.DataFrame({'overlap_perturbation_gene': overlap})

,overlap_perturbation_gene
0,ACTB
1,ANKRD11
2,ATP2A2
3,C17orf58
4,CSE1L
5,CYS1
6,DLD
7,DNMT1
8,EEF2
9,EIF1


In [5]:
pd.DataFrame({'perturbation_missing_from_var': missing_from_var})

,perturbation_missing_from_var
0,ALG14
1,CIAPIN1
2,CLASRP
3,CNOT3
4,CPSF3
5,CSNK2B
6,DDX46
7,E4F1
8,EIF2B1
9,EIF2B2


In [6]:
cell_counts = adata.obs['perturbation'].astype(str).value_counts()

overlap_counts = (
    cell_counts.loc[overlap]
    .rename_axis('perturbation')
    .reset_index(name='n_cells')
    .sort_values('n_cells', ascending=False)
)
overlap_counts

,perturbation,n_cells
11,GFM1,6039
1,ANKRD11,3918
24,MRPL36,3814
41,TFAM,3627
19,JAZF1,3363
13,GRSF1,3221
38,SIN3A,3181
31,PSMB5,2722
36,RPL41,2669
33,PTMA,2511


In [7]:
adata.obs['perturbation_type'].astype(str).value_counts(dropna=False)

perturbation_type
CRISPRi    197249
Name: count, dtype: int64

In [8]:
adata.file.close()